# Retention Policy

Exploring more about Lifecycle data with AWS S3:

- [Compreender e gerenciar classes de armazenamento do Amazon S3](https://docs.aws.amazon.com/pt_br/AmazonS3/latest/userguide/storage-class-intro.html)
- [Transição de objetos usando o Amazon S3 Lifecycle](https://docs.aws.amazon.com/pt_br/AmazonS3/latest/userguide/lifecycle-transition-general-considerations.html)
- [AWS S3 - Muito mais do que um Simples Serviço de Armazenamento](https://blog.dsbrigade.com/introducao-aws-s3/)


In [ ]:
import json
import boto3
import json
from botocore.exceptions import ClientError

# Constants for Days and Durations
DAYS_TO_INFREQUENT_ACCESS = 30
DAYS_TO_GLACIER = 90
DAYS_TO_EXPIRE = 3650  # 10 years
NONCURRENT_VERSION_DAYS = 30
NONCURRENT_VERSION_EXPIRATION_DAYS = 365
ABORT_MULTIPART_UPLOAD_DAYS = 7
BUCKET_BRONZE = "bronze"

In [ ]:
# Define the lifecycle policy using constants
lifecycle_policy = {
    "Rules": [
        {
            "ID": "TransitionToInfrequentAccess",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled",
            "Transitions": [
                {
                    "Days": DAYS_TO_INFREQUENT_ACCESS,
                    "StorageClass": "STANDARD_IA"
                }
            ],
            "NoncurrentVersionTransitions": [
                {
                    "NoncurrentDays": NONCURRENT_VERSION_DAYS,
                    "StorageClass": "STANDARD_IA"
                }
            ],
            "NoncurrentVersionExpiration": {
                "NoncurrentDays": NONCURRENT_VERSION_EXPIRATION_DAYS
            },
            "AbortIncompleteMultipartUpload": {
                "DaysAfterInitiation": ABORT_MULTIPART_UPLOAD_DAYS
            }
        },
        {
            "ID": "TransitionToGlacier",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled",
            "Transitions": [
                {
                    "Days": DAYS_TO_GLACIER,
                    "StorageClass": "GLACIER"
                }
            ],
            "NoncurrentVersionTransitions": [
                {
                    "NoncurrentDays": DAYS_TO_GLACIER,
                    "StorageClass": "GLACIER"
                }
            ]
        },
        {
            "ID": "ExpireObjects",
            "Filter": {
                "Prefix": ""
            },
            "Status": "Enabled",
            "Expiration": {
                "Days": DAYS_TO_EXPIRE
            },
            "NoncurrentVersionExpiration": {
                "NoncurrentDays": DAYS_TO_EXPIRE
            }
        }
    ]
}

# Write the lifecycle policy to a JSON file
with open('lifecycle.json', 'w') as f:
    json.dump(lifecycle_policy, f, indent=2)

print("Lifecycle policy JSON file generated.")
lifecycle_policy


In [ ]:
# Inicializar o cliente S3
s3_client = boto3.client('s3')

# Aplicar a política de ciclo de vida
try:
    s3_client.put_bucket_lifecycle_configuration(
        Bucket=BUCKET_BRONZE,
        LifecycleConfiguration=lifecycle_policy
    )
    print("Política de ciclo de vida aplicada com sucesso ao bucket '{}'.".format(BUCKET_BRONZE))
except ClientError as e:
    print(f"Erro ao aplicar a política de ciclo de vida: {e}")

In [ ]:
try:
    response = s3_client.get_bucket_lifecycle_configuration(Bucket=BUCKET_BRONZE)
    print("Configuração de ciclo de vida para o bucket '{}':".format(BUCKET_BRONZE))
    print(json.dumps(response, indent=2))
except s3_client.exceptions.NoSuchLifecycleConfiguration:
    print("Nenhuma configuração de ciclo de vida encontrada para o bucket '{}'.".format(BUCKET_BRONZE))
except ClientError as e:
    print(f"Erro ao obter a configuração de ciclo de vida: {e}")
